## Custom dataset:
In pytorch datasets are classes

In [29]:
import torch 
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self):
        self.x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
        self.y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])

    def __len__(self):
        #How big dataset is
        return len(self.x)

    def _getitem__(self, index):
        #to get one sample
        return self.x[index], self.y[index]

## DataLoader:
DataLoader loads data in batches.

In [30]:
from torch.utils.data import DataLoader

dataset = MyDataset()
loader = DataLoader(dataset, batch_size=2, shuffle=True)

now our training loop becomes:

    for epoch in range(1000):
        for x_batch, y_batch in loader:

            y_pred = model(x_batch)
            loss = loss_fn(y_pred, y_batch)
    
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

This can also be done without using a custom class, by using a util called **TensorDataset**

In [31]:
from torch.utils.data import TensorDataset

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])

dataset = TensorDataset(x,y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [32]:
for x_batch, y_batch in loader:
    print(x_batch,'\n', y_batch)

tensor([[2.],
        [1.]]) 
 tensor([[5.],
        [3.]])
tensor([[3.],
        [4.]]) 
 tensor([[7.],
        [9.]])


If you are using csv data then for x and y you can use this on a pandas dataframe:

    X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)
    y = torch.tensor(df.iloc[:, -1].values, dtype=torch.float32).unsqueeze(1)

Lets try doing this on a tabular dataset:

In [33]:
import pandas as pd

dataset = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
dataset.isna().sum()

id                  0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

theres no null values, lets see if there are any cartegorical features that need to be converted to numerical through encoding:

In [34]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [38]:
#performing encoding using pandas
df = pd.get_dummies(dataset, drop_first=True)

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 32 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   id                                     594194 non-null  int64  
 1   SeniorCitizen                          594194 non-null  int64  
 2   tenure                                 594194 non-null  int64  
 3   MonthlyCharges                         594194 non-null  float64
 4   TotalCharges                           594194 non-null  float64
 5   gender_Male                            594194 non-null  bool   
 6   Partner_Yes                            594194 non-null  bool   
 7   Dependents_Yes                         594194 non-null  bool   
 8   PhoneService_Yes                       594194 non-null  bool   
 9   MultipleLines_No phone service         594194 non-null  bool   
 10  MultipleLines_Yes                      594194 non-null  

In [44]:
import numpy as np

x = torch.tensor(df.iloc[:, :-1].to_numpy(dtype=np.float32)) #all rows, all columns except last one 
y = torch.tensor(df.iloc[:, -1].to_numpy(dtype=np.float32)).unsqueeze(1) #all rows but only last column 

dataset = TensorDataset(x,y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [46]:
for x_batch,y_batch in loader:
    print(x_batch,'\n',y_batch)
    break

tensor([[3.9697e+05, 0.0000e+00, 2.4000e+01, 5.0100e+01, 1.2082e+03, 1.0000e+00,
         1.0000e+00, 1.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         1.0000e+00],
        [1.6510e+05, 1.0000e+00, 2.4000e+01, 8.0750e+01, 1.7489e+03, 1.0000e+00,
         0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00, 1.0000e+00,
         0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
         0.0000e+00],
        [3.5348e+04, 0.0000e+00, 1.0000e+00, 4.4950e+01, 4.4950e+01, 0.0000e+00,
         1.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+